# 04 - Reliability: Sensitivity to Small Input Noise

Tests whether a rating is stable under input perturbations too small to reflect a genuine difference in performance: the off-by-one ambiguity
that can survive user validation (22 vs 23 passes from a slightly unclear screenshot). Two real performances differing only by that kind of noise
should get the same, or very nearly the same, rating.

**Design (fixed before running).**
- Agreement is **ICC(A,1)** (McGraw and Wong; Shrout and Fleiss's ICC2): two-way random effects, absolute agreement, single measure, with a
  percentile bootstrap over performances (`workshop/lib/reliability.py`, checked against pingouin by a test).
- Targets are real rated performances from every career; raters are **1 original + 10 noisy variants** of each.
- Noise is **additive**: one of `{-1, 0, +1}` on the integer counts (passes, dribbles, tackles, possession won and lost, fouls, offsides, shots) and
  one of `{-0.1, 0, +0.1}` on the decimal distance fields; values never go below 0. Goals, assists, percentages and minutes are untouched.
  (Proportional noise was rejected: +/-7.5% of a small integer rounds back to itself, and anything at 0 is immune to it.)
- Interpretation bands (Koo and Li, 2016): below 0.5 poor, 0.5 to 0.75 moderate, 0.75 to 0.9 good, above 0.9 excellent.

**Extensions.** A noise-magnitude sweep; reliability by position group; goalkeepers (noise on saves, shots on target and shots against); and a
near-neighbour retest (do performances that are close in stat space get close ratings?).

Half-length invariance is covered by the data tests, not here: it is a single scalar multiplication in the service.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from workshop.lib import crossval, datasets, decompose, fit, gate, inference, rating, refit, sweep, variants
from workshop.lib.report import ci, header, say

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
from sklearn.neighbors import NearestNeighbors
from workshop.lib import reliability

In [2]:
info = header()

**Validated config: `005d563f`** | run 2026-10-25

- Config fitted on: arsenal_2 1-38, ipswich_3 1-22, valencia_cf_1 1-157
- Data analysed (matches per career): arsenal_2 (38), ipswich_3 (22), valencia_cf_1 (157)

In [3]:
SEED = 42
N_NOISY = 10
N_BOOT = 1000
MIN_GROUP_ROWS = 30
svc = rating.load_service()
POSITION_GROUP_MAP = {
    "ST": "ST", "CF": "ST", "LW": "Winger", "RW": "Winger", "LM": "WM", "RM": "WM",
    "CAM": "CAM", "CM": "CM", "CDM": "CDM", "CB": "CB", "LB": "Fullback", "RB": "Fullback",
}

records = []
for career in datasets.load_careers("pool"):
    for match in career.matches:
        for perf in match["player_performances"]:
            original = rating.rate_performance(svc, career.club, match["data"], perf)
            if original is None:
                continue
            is_gk = perf["performance_type"] == "GK"
            first = "GK" if is_gk else perf["positions_played"][0]
            single = is_gk or len(perf["positions_played"]) == 1
            records.append({"career": career.name, "club": career.club, "data": match["data"], "perf": perf,
                            "gk": is_gk, "group": "GK" if is_gk else POSITION_GROUP_MAP.get(first), "single": single,
                            "original": original})
outfield = [r for r in records if not r["gk"]]
keepers = [r for r in records if r["gk"]]
print(f"{len(outfield)} rated outfield performances, {len(keepers)} goalkeeper performances")

3042 rated outfield performances, 217 goalkeeper performances


## Noise on outfield performances

In [4]:
def rating_matrix(rows, count_step=1, distance_step=0.1, count_stats=reliability.COUNT_STATS,
                  distance_stats=reliability.DISTANCE_STATS, seed=SEED):
    """Rows x (1 original + N_NOISY noisy) matrix of ratings."""
    rng = np.random.default_rng(seed)
    out = np.empty((len(rows), N_NOISY + 1))
    for i, r in enumerate(rows):
        out[i, 0] = r["original"]
        for j in range(1, N_NOISY + 1):
            noisy = reliability.perturb(r["perf"], rng, count_step, distance_step, count_stats, distance_stats)
            out[i, j] = rating.rate_performance(svc, r["club"], r["data"], noisy)
    return out


def noise_summary(matrix: np.ndarray) -> dict[str, float]:
    diff = np.abs(matrix[:, 1:] - matrix[:, [0]])
    return {
        "mean_abs_change": float(diff.mean()),
        "share_unchanged": float((diff == 0).mean()),
        "share_within_0_1": float((diff <= 0.1 + 1e-9).mean()),
        "share_moved_0_5": float((diff >= 0.5 - 1e-9).mean()),
        "max_abs_change": float(diff.max()),
        "rank_corr_with_original": float(np.mean([stats.spearmanr(matrix[:, 0], matrix[:, j]).statistic for j in range(1, N_NOISY + 1)])),
    }


main = rating_matrix(outfield)
icc = reliability.bootstrap_icc_a1(main, N_BOOT, SEED)
outfield_summary = noise_summary(main)
display(pd.DataFrame({"ICC(A,1)": [icc.estimate], "ci_low": [icc.ci_low], "ci_high": [icc.ci_high], "band": [reliability.band(icc.estimate)],
                      **{k: [v] for k, v in outfield_summary.items()}}, index=["outfield, +/-1 count, +/-0.1 distance"]))

,"ICC(A,1)",ci_low,ci_high,band,mean_abs_change,share_unchanged,share_within_0_1,share_moved_0_5,max_abs_change,rank_corr_with_original
"outfield, +/-1 count, +/-0.1 distance",0.929,0.924,0.934,excellent,0.189,0.227,0.561,0.099,1.900,0.941


In [5]:
say(
    f"**Outfield reliability.** With +/-1 on counts and +/-0.1 on distances, ICC(A,1) = **{icc.estimate:.3f}** "
    f"[{icc.ci_low:.3f}, {icc.ci_high:.3f}] ({reliability.band(icc.estimate)}). A noisy variant's rating equals the original in "
    f"{outfield_summary['share_unchanged']:.0%} of cases, is within 0.1 in {outfield_summary['share_within_0_1']:.0%}, moves by half a point or more in "
    f"{outfield_summary['share_moved_0_5']:.1%} (largest {outfield_summary['max_abs_change']:.1f}), and averages {outfield_summary['mean_abs_change']:.3f} "
    f"away; rank correlation with the original is {outfield_summary['rank_corr_with_original']:.3f}."
)

**Outfield reliability.** With +/-1 on counts and +/-0.1 on distances, ICC(A,1) = **0.929** [0.924, 0.934] (excellent). A noisy variant's rating equals the original in 23% of cases, is within 0.1 in 56%, moves by half a point or more in 9.9% (largest 1.9), and averages 0.189 away; rank correlation with the original is 0.941.

## Noise magnitude

How fast does agreement fall as the misreading grows? Counts move by `k` and distances by `0.1k`.

In [6]:
rows = []
matrices = {1: main}
for k in (1, 2, 3, 5):
    m = matrices[k] if k in matrices else rating_matrix(outfield, count_step=k, distance_step=0.1 * k, seed=SEED + k)
    matrices[k] = m
    est = reliability.bootstrap_icc_a1(m, 300, SEED)
    rows.append({"noise k": k, "ICC(A,1)": est.estimate, "ci_low": est.ci_low, "ci_high": est.ci_high,
                 "band": reliability.band(est.estimate), **noise_summary(m)})
sweep_table = pd.DataFrame(rows).set_index("noise k")
sweep_table

,"ICC(A,1)",ci_low,ci_high,band,mean_abs_change,share_unchanged,share_within_0_1,share_moved_0_5,max_abs_change,rank_corr_with_original
noise k,,,,,,,,,,
1,0.929,0.924,0.934,excellent,0.189,0.227,0.561,0.099,1.900,0.941
2,0.814,0.802,0.824,good,0.325,0.142,0.378,0.248,2.500,0.860
3,0.705,0.690,0.719,moderate,0.440,0.105,0.293,0.354,3.300,0.783
5,0.550,0.532,0.568,moderate,0.621,0.075,0.209,0.491,3.700,0.670


In [7]:
drop = sweep_table["ICC(A,1)"]
first_below = next((k for k, v in drop.items() if v < 0.9), None)
say(
    "**Noise sweep.** ICC(A,1) by noise size k: " + ", ".join(f"k={k}: {v:.3f}" for k, v in drop.items())
    + (f". Agreement first drops out of the 'excellent' band at k = {first_below}." if first_below else ". Agreement stays 'excellent' across the sweep.")
)

**Noise sweep.** ICC(A,1) by noise size k: k=1: 0.929, k=2: 0.814, k=3: 0.705, k=5: 0.550. Agreement first drops out of the 'excellent' band at k = 2.

## By position group (outfield)

In [8]:
rows = []
for group in ["ST", "Winger", "WM", "CAM", "CM", "CDM", "CB", "Fullback"]:
    idx = [i for i, r in enumerate(outfield) if r["group"] == group and r["single"]]
    if len(idx) < MIN_GROUP_ROWS:
        rows.append({"group": group, "rows": len(idx)})
        continue
    sub = main[idx]
    est = reliability.bootstrap_icc_a1(sub, 300, SEED)
    rows.append({"group": group, "rows": len(idx), "ICC(A,1)": est.estimate, "ci_low": est.ci_low, "ci_high": est.ci_high,
                 "band": reliability.band(est.estimate), **{k: v for k, v in noise_summary(sub).items() if k in ("mean_abs_change", "share_moved_0_5")}})
by_group = pd.DataFrame(rows).set_index("group")
by_group

,rows,"ICC(A,1)",ci_low,ci_high,band,mean_abs_change,share_moved_0_5
group,,,,,,,
ST,322,0.965,0.957,0.972,excellent,0.163,0.068
Winger,546,0.965,0.957,0.971,excellent,0.142,0.049
WM,69,0.919,0.853,0.955,excellent,0.129,0.032
CAM,29,NaN,NaN,NaN,NaN,NaN,NaN
CM,536,0.924,0.907,0.936,excellent,0.178,0.093
CDM,308,0.902,0.882,0.918,excellent,0.209,0.123
CB,560,0.887,0.871,0.902,good,0.246,0.158
Fullback,562,0.909,0.894,0.920,excellent,0.201,0.106


In [9]:
tested = by_group.dropna(subset=["ICC(A,1)"])
low = tested["ICC(A,1)"].idxmin()
say(
    "**By position.** ICC(A,1): " + ", ".join(f"{g} {r['ICC(A,1)']:.3f}" for g, r in tested.iterrows())
    + f". Lowest: **{low}** ({tested.loc[low, 'ICC(A,1)']:.3f}, {tested.loc[low, 'band']}). "
    + (f"Not tested (under {MIN_GROUP_ROWS} rows): {', '.join(by_group.index[by_group['ICC(A,1)'].isna()])}." if by_group["ICC(A,1)"].isna().any() else "")
)

**By position.** ICC(A,1): ST 0.965, Winger 0.965, WM 0.919, CM 0.924, CDM 0.902, CB 0.887, Fullback 0.909. Lowest: **CB** (0.887, good). Not tested (under 30 rows): CAM.

## Goalkeepers

Noise of +/-1 on saves, shots on target and shots against (the stats a misread screenshot would change); goals conceded and the save rate are left alone.

In [10]:
gk_main = rating_matrix(keepers, count_stats=reliability.GK_COUNT_STATS, distance_stats=())
gk_icc = reliability.bootstrap_icc_a1(gk_main, N_BOOT, SEED)
gk_summary = noise_summary(gk_main)
rows = [{"noise k": 1, "ICC(A,1)": gk_icc.estimate, "ci_low": gk_icc.ci_low, "ci_high": gk_icc.ci_high, "band": reliability.band(gk_icc.estimate), **gk_summary}]
for k in (2, 3):
    m = rating_matrix(keepers, count_step=k, count_stats=reliability.GK_COUNT_STATS, distance_stats=(), seed=SEED + k)
    est = reliability.bootstrap_icc_a1(m, 300, SEED)
    rows.append({"noise k": k, "ICC(A,1)": est.estimate, "ci_low": est.ci_low, "ci_high": est.ci_high, "band": reliability.band(est.estimate), **noise_summary(m)})
gk_table = pd.DataFrame(rows).set_index("noise k")
gk_table

,"ICC(A,1)",ci_low,ci_high,band,mean_abs_change,share_unchanged,share_within_0_1,share_moved_0_5,max_abs_change,rank_corr_with_original
noise k,,,,,,,,,,
1,0.984,0.979,0.988,excellent,0.122,0.378,0.666,0.034,1.000,0.987
2,0.944,0.928,0.956,excellent,0.243,0.323,0.430,0.175,1.300,0.959
3,0.902,0.876,0.922,excellent,0.340,0.313,0.384,0.342,1.600,0.930


In [11]:
say(
    f"**Goalkeepers ({len(keepers)} performances).** ICC(A,1) = **{gk_icc.estimate:.3f}** [{gk_icc.ci_low:.3f}, {gk_icc.ci_high:.3f}] "
    f"({reliability.band(gk_icc.estimate)}) at +/-1; a noisy variant is unchanged in {gk_summary['share_unchanged']:.0%} of cases and moves by half a "
    f"point or more in {gk_summary['share_moved_0_5']:.1%} (largest {gk_summary['max_abs_change']:.1f}). By noise size: "
    + ", ".join(f"k={k}: {v:.3f}" for k, v in gk_table["ICC(A,1)"].items()) + "."
)

**Goalkeepers (217 performances).** ICC(A,1) = **0.984** [0.979, 0.988] (excellent) at +/-1; a noisy variant is unchanged in 38% of cases and moves by half a point or more in 3.4% (largest 1.0). By noise size: k=1: 0.984, k=2: 0.944, k=3: 0.902.

## Near-neighbour retest

Reliability under noise is one view of stability; the other is whether performances that are *close in stat space* get close ratings. Within each position
group (single-position, 30+ rows), every performance is paired with its nearest neighbour in standardised stat space, and the rating gap of those pairs is
compared with the gap of randomly paired performances.

In [12]:
FEATURES = ["passes", "pass_accuracy", "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides",
            "fouls_committed", "possession_won", "possession_lost", "distance_covered", "distance_sprinted", "shots", "shot_accuracy",
            "goals", "assists"]
rng = np.random.default_rng(SEED)
rows = []
for group in ["ST", "Winger", "WM", "CM", "CDM", "CB", "Fullback"]:
    part = [r for r in outfield if r["group"] == group and r["single"]]
    if len(part) < MIN_GROUP_ROWS:
        continue
    x = pd.DataFrame([{f: r["perf"].get(f, 0) for f in FEATURES} for r in part]).astype(float)
    x = (x - x.mean()) / x.std().replace(0, 1)
    y = np.array([r["original"] for r in part])
    neighbours = NearestNeighbors(n_neighbors=2).fit(x.values).kneighbors(x.values, return_distance=False)[:, 1]
    near = np.abs(y - y[neighbours])
    shuffled = np.abs(y - y[rng.permutation(len(y))])
    rows.append({"group": group, "rows": len(part), "mean |gap| nearest neighbour": near.mean(), "mean |gap| random pair": shuffled.mean(),
                 "ratio random / nearest": shuffled.mean() / near.mean(), "share of neighbours within 0.1": (near <= 0.1 + 1e-9).mean()})
neighbour = pd.DataFrame(rows).set_index("group")
neighbour

,rows,mean |gap| nearest neighbour,mean |gap| random pair,ratio random / nearest,share of neighbours within 0.1
group,,,,,
ST,322,0.453,1.176,2.593,0.314
Winger,546,0.421,0.997,2.367,0.306
WM,69,0.371,0.519,1.398,0.232
CM,536,0.416,0.886,2.129,0.304
CDM,308,0.494,0.894,1.812,0.227
CB,560,0.497,0.940,1.892,0.255
Fullback,562,0.487,0.888,1.824,0.265


In [13]:
say(
    "**Near neighbours.** Pairs of performances that are closest in stat space differ by "
    + ", ".join(f"{g} {r['mean |gap| nearest neighbour']:.2f}" for g, r in neighbour.iterrows())
    + " rating points on average, against "
    + ", ".join(f"{g} {r['mean |gap| random pair']:.2f}" for g, r in neighbour.iterrows())
    + f" for random pairs (a {neighbour['ratio random / nearest'].min():.1f}x to {neighbour['ratio random / nearest'].max():.1f}x difference)."
)

**Near neighbours.** Pairs of performances that are closest in stat space differ by ST 0.45, Winger 0.42, WM 0.37, CM 0.42, CDM 0.49, CB 0.50, Fullback 0.49 rating points on average, against ST 1.18, Winger 1.00, WM 0.52, CM 0.89, CDM 0.89, CB 0.94, Fullback 0.89 for random pairs (a 1.4x to 2.6x difference).

## Notes and limitations

- Noise is applied to inputs one stat at a time but independently across stats, so it does not model correlated misreads (a whole row shifted).
- Percentages are not perturbed: if the OCR misreads a count, the percentage the screen shows is still correct.
- Only rated performances are included (outfield players with 10+ minutes), as the service declines to rate the rest.
- Rating reliability is about stability, not validity: a rating can be perfectly reproducible and still wrong (see `01_construct_validity`).

In [14]:
say(
    f"**Summary (config `{info['hash'][:8]}`).** Outfield ICC(A,1) {icc.estimate:.2f} ({reliability.band(icc.estimate)}), goalkeepers {gk_icc.estimate:.2f} "
    f"({reliability.band(gk_icc.estimate)}); lowest position {low} ({tested.loc[low, 'ICC(A,1)']:.2f}); outfield agreement is "
    + ("'excellent' up to k = " + str(max(k for k, v in drop.items() if v >= 0.9)) if (drop >= 0.9).any() else "below 'excellent' even at k = 1") + "."
)

**Summary (config `005d563f`).** Outfield ICC(A,1) 0.93 (excellent), goalkeepers 0.98 (excellent); lowest position CB (0.89); outfield agreement is 'excellent' up to k = 1.